### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [16]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [17]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

### Data generator

##### Support functions

In [18]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [19]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.7, 1.5), (0.6, 0.7, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [20]:
############################### Function to generate the credit to be requested ##################################
# Note the credits will be exactly for one year
def generate_credit_requested(income): ### It will depend on the income
    # The will maximum enter as for a credit that represent between 50% and 200% of their income and the probabilities of any values are equal, so a uniform distribution
    percentage_of_monthly_income = random.uniform(0.5, 2.0)
    yearly_income = income * 12 # must be changes once dynamically made
    credit_requested_yearly = yearly_income *percentage_of_monthly_income
    credit_requested_monthly = credit_requested_yearly / 12
    percentage_credit_month_income = credit_requested_monthly/income
    return credit_requested_monthly, percentage_credit_month_income
    

##### Data Generator for the original state of individuals

In [ ]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        
        # Debt to income ratio: PARTIAL, before the credit
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio_partial = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio_partial = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        monthly_credit = generate_credit_requested(monthly_income)[0]
        credit_to_income_ratio = generate_credit_requested(monthly_income)[1]
        
        # calculating the monthly debt if the monthluy credit is issued
        debt_after_credit = savings_debt - monthly_credit
        if debt_after_credit > 0: # if still the monthly savings are higher than the credit, then the total debt to incom ratio should be zero
            debt_to_income_ratio_total = 0
        else: 
            debt_to_income_ratio_total = abs(debt_after_credit/monthly_income)
        
        # ---------------- Y-Variable --------------------------------#
        # PD, LGD, EADs
        
        data.append({
            'name': name, # independent
            'age': age, # independent
            'educational level': education_level, # independent
            'number of not paid past credits': past_credits, # independent
            'dependents': dependents, # independent
            'profession': profession, # depends on education
            'monthly income': monthly_income, # depends on profession and age
            'monthly expenditure': monthly_expenditure, # depends on income, mean income per age and profession, and the number of dependents
            'savings (debt)': savings_debt, # monthly income - monthly expenditure
            'debt-to-income ratio before credit': debt_to_income_ratio_partial, # abs(savings_debt/monthly_income)
            'credit: monthly amount': monthly_credit, # depends on the income
            'credit-to-income ratio': credit_to_income_ratio, # credit/income
            'debt-to-income ratio after credit': debt_to_income_ratio_total # abs((savings_debt - credit)/monthly_income)
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [23]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,debt-to-income ratio after credit
0,name0,53,high school or lower,0,0,LowSkilled,2028.0,1840.870758,187.129242,0.000000,3951.171005,1.341469,1.856036
1,name1,50,ausbildung,0,0,MediumSkilled,2359.0,1813.340538,545.659462,0.000000,2656.873948,0.831060,0.894962
2,name2,31,bachelor degree,0,0,Unemployed_HighSkilled,1500.0,1332.009418,167.990582,0.000000,1868.962331,0.597417,1.133981
3,name3,30,ausbildung,0,3,MediumSkilled,1374.0,2767.924362,-1393.924362,1.014501,1622.330685,0.738750,2.195237
4,name4,35,ausbildung,0,0,MediumSkilled,2082.0,1609.653586,472.346414,0.000000,1966.147755,1.850588,0.717484
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,53,ausbildung,1,0,MediumSkilled,3989.0,2235.004378,1753.995622,0.000000,6546.859094,1.959538,1.201520
996,name996,57,ausbildung,0,1,MediumSkilled,4476.0,3035.847168,1440.152832,0.000000,3667.940717,0.692200,0.497718
997,name997,36,bachelor degree,0,1,HighSkilled,3968.0,6186.126508,-2218.126508,0.559004,6796.626316,1.140268,2.271863
998,name998,33,ausbildung,1,1,MediumSkilled,1874.0,1334.641196,539.358804,0.000000,2443.885829,0.594268,1.016290


##### DF statistics

In [24]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,debt-to-income ratio after credit
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.646000,0.574000,0.547000,3558.102000,3354.984249,203.117751,0.119587,4449.932857,1.251715,1.230555
std,8.758733,0.734209,0.816368,2548.190616,2559.259000,1618.424375,0.243467,3669.449878,0.446550,0.545735
min,30.000000,0.000000,0.000000,500.000000,370.792973,-8994.494165,0.000000,321.334767,0.501692,0.000000
25%,37.000000,0.000000,0.000000,1747.250000,1585.778060,-323.610402,0.000000,1965.678097,0.868514,0.838203
50%,45.000000,0.000000,0.000000,2770.500000,2433.216942,185.348546,0.000000,3242.731015,1.267028,1.225480
75%,52.000000,1.000000,1.000000,4531.250000,4467.933235,758.948037,0.134467,5580.049505,1.655325,1.600801
max,60.000000,3.000000,4.000000,15533.000000,16741.260418,9779.720932,2.426354,22892.980628,1.999139,3.927694


Monthly income by profession

In [25]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,358.0,6046.865922,2591.901469,1939.0,3966.50,5714.5,7680.75,15533.0
LowSkilled,270.0,1571.907407,488.867802,509.0,1206.75,1546.0,1895.00,2806.0
MediumSkilled,290.0,2836.065517,907.543168,1150.0,2089.50,2803.5,3385.75,5507.0
Unemployed_HighSkilled,35.0,2685.714286,1029.603824,1500.0,1500.00,2250.0,3250.00,4500.0
Unemployed_LowSkilled,24.0,808.333333,209.899907,500.0,600.00,900.0,1000.00,1100.0
Unemployed_MediumSkilled,23.0,1436.956522,412.645950,900.0,1150.00,1300.0,1875.00,2000.0


Debt-to-income ratio by profession

In [26]:
df_1.groupby('profession')['debt-to-income ratio before credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,358.0,0.152254,0.306661,0.0,0.0,0.000000,0.178667,2.426354
LowSkilled,270.0,0.111851,0.200232,0.0,0.0,0.000000,0.133937,0.945539
MediumSkilled,290.0,0.105668,0.214679,0.0,0.0,0.000000,0.111620,1.319813
Unemployed_HighSkilled,35.0,0.060813,0.092146,0.0,0.0,0.018975,0.102351,0.437949
Unemployed_LowSkilled,24.0,0.060619,0.096133,0.0,0.0,0.000000,0.137321,0.286457
Unemployed_MediumSkilled,23.0,0.028410,0.092760,0.0,0.0,0.000000,0.000000,0.427750


In [27]:
df_1.groupby('profession')['debt-to-income ratio after credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,358.0,1.249638,0.593044,0.000000,0.833691,1.260276,1.646032,3.927694
LowSkilled,270.0,1.270930,0.502946,0.081037,0.960242,1.283941,1.590691,2.722064
MediumSkilled,290.0,1.202006,0.547727,0.105478,0.830353,1.139253,1.598323,2.753093
Unemployed_HighSkilled,35.0,1.218626,0.434638,0.539923,0.821406,1.220847,1.596219,2.048861
Unemployed_LowSkilled,24.0,1.070919,0.470136,0.246425,0.799681,0.990649,1.350473,1.890479
Unemployed_MediumSkilled,23.0,1.004267,0.376422,0.241121,0.679466,1.020021,1.267375,1.644594
